In [47]:
import json

import numpy as np
import pandas as pd

In [ ]:
def load_phenotype(name):
    # Loads an ABCD phenotype file and replaces numeric codes with labels.
    # Expects a paired .tsv (data) and .json (data dictionary) at the same path.
    # The JSON uses BIDS-style "Levels" dicts that map integer codes to string labels
    # (e.g. {"1": "Scanner", "2": "Laptop"}) — these are applied column-by-column.
    base = f'/mnt/d/OneDrive/Data/ABCD/v7/{name}'
    with open(f'{base}.json') as f:
        data_dict = json.load(f)
    df = pd.read_csv(f'{base}.tsv', sep='\t')
    for col, meta in data_dict.items():
        if col in df.columns and 'Levels' in meta:
            # str(int(x)) normalizes float-coded integers (e.g. 1.0 → "1") before lookup.
            # levels=meta['Levels'] binds the current value at definition time so the
            # closure can't capture a later loop iteration's meta (ruff B023).
            df[col] = df[col].map(
                lambda x, levels=meta['Levels']: levels.get(str(int(x)), x) if pd.notna(x) else x
            )
    return df


mr_info = load_phenotype('mr_y_adm__info')  # MR session administrative info

In [49]:
# From Hagler et al. p. 8, footnote 2:
# A total of 16 initial frames (12.8 s) are discarded. On Siemens and Philips
# scanners, the first eight frames make up the pre-scan reference, and are not
# saved as DICOMS. An additional eight frames are discarded as part of the pre-
# analysis processing, for a total of 16 initial frames. On GE scanners with soft-
# ware version DV25, the first 12 frames make up the pre-scan reference. Instead
# of being discarded, those 12 reference scans are combined into one, and saved as
# the first frame, for a total of five initial frames to be discarded as part of the pre-
# analysis processing for GE DV25 series. On GE scanners with software version
# DV26, the pre-scan reference is not retained at all, and a total of 16 initial frames
# are discarded for GE DV26 scans as part of the pre-analysis processing

In [50]:
# Get the GE DV (or RX) version out of the software version string
# mr_info['GE_DV_ver'] = mr_info['mr_y_adm__info__dev__sftw_ver'].str.extract(r"[DMR][VRX](\d{2}).*").astype('Int64')

mr_info['GE_DV_ver'] = (
    mr_info['mr_y_adm__info__dev__sftw_ver']
    .str.extract(r"[DMR][VRX](\d{2})", expand=False)
    .astype('Int64')
)
# Create conditions for numpy select recoding
conditions = [
    (mr_info['mr_y_adm__info__dev_manufact'] == 'Siemens'),
    (mr_info['mr_y_adm__info__dev_manufact'] == 'Philips'),
    (mr_info['mr_y_adm__info__dev_manufact'] == 'GE') & (mr_info['GE_DV_ver'] == 25).fillna(False),
    (mr_info['mr_y_adm__info__dev_manufact'] == 'GE') & (mr_info['GE_DV_ver'] > 25).fillna(False),
]

# Choices to match conditions (from Hagler et al., 2016, footnote 4)
choices = [
    # Discard eight frames for Siemens and Philips scanners
    8,
    8,
    # Discard 5 frames for GE scanners with DV 25
    5,
    # Discard 16 frames for GE scanners with DV > 25
    16,
]

In [51]:
mr_info
mr_info['nss_frames'] = np.select(conditions, choices, default=None)
mr_info = mr_info[
    ['participant_id', 'session_id', 'nss_frames', 'mr_y_adm__info__dev_manufact', 'GE_DV_ver']
]
mr_info = mr_info.rename(
    columns={
        "participant_id": "subject_id",
        "session_id": "session",
        "mr_y_adm__info__dev_manufact": "scanner_make",
        "GE_DV_ver": "GEmajDVver",
    }
)

In [52]:
mr_info.to_csv('nss_volumes.csv')